In [ ]:
from torch.nn import *
import torch
import numpy as np
from sklearn.datasets import make_classification

torch.set_default_tensor_type(torch.cuda.FloatTensor)


def train(x, y, log=0):
    m = Sequential(Linear(4, 4), Tanh(), Linear(4, 2))
    o = torch.optim.Adam(m.parameters(), 0.1, weight_decay=0.03)
    for _ in range(5):
        for _ in range(100):
            _ = functional.cross_entropy(m(x), y)
            _.backward()
            o.step()
            o.zero_grad()
        if log:
            print(_)
    return m


def uniq(m):
    with torch.no_grad():
        sign = m[0].bias.sign()
        sort = m[0].bias.mul_(sign).argsort()
        m[0].weight = Parameter(m[0].weight.mul_(sign[:, None])[sort])
        m[0].bias = Parameter(m[0].bias[sort])
        m[2].weight = Parameter(m[2].weight.mul_(sign)[:, sort])
        return m


def param(m, *a):
    with torch.no_grad():
        return torch.cat((*(i.detach().ravel() for i in m.parameters()), *a))


def unparam(a):
    a = a.split((16, 4, 8, 2))
    m = Sequential(Module(), Tanh(), Module())
    m[0].weight = Parameter(a[0].view(4, 4))
    m[0].bias = Parameter(a[1])
    m[2].weight = Parameter(a[2].view(2, 4))
    m[2].bias = Parameter(a[3])
    m[0].__class__ = m[2].__class__ = Linear
    return m


def xyrRXY():
    x, y = make_classification(np.random.randint(1, 16), 4)
    x = torch.tensor(x, dtype=torch.float)
    y = torch.tensor(y, dtype=torch.long)
    m = uniq(train(x[:-1], y[:-1]))
    M = uniq(train(x, y))
    X = x[-1]
    Y = y[-1]
    return (
        param(m, X, Y[None].float()),
        param(M),
        int(m(X).argmax() == Y),
        int(M(X).argmax() == Y),
        X,
        Y,
    )

In [ ]:
# m = Sequential(Linear(35, 256), Hardswish(1), Linear(256, 30))
# o = torch.optim.Adam(m.parameters(), 1e-4)
while 1:
    before = 0
    after = 0
    infer = 0
    for _ in range(100):
        x, y, r, R, X, Y = xyrRXY()
        before += r
        after += R
        _ = m(x)
        infer += int(unparam(_)(X).argmax() == Y)
        _ = functional.smooth_l1_loss(_, y)
        _.backward()
        o.step()
        o.zero_grad()
    print(_, before, after, infer)
torch.save(m, "nnTransferer.model.pk")

# OLD

In [ ]:
from pickle import load

a = load(open(f"nnTransferer.pk", "rb"))
np.stack(
    [
        np.concatenate((np.stack(a[2][:-1]), a[0][1:], a[1][1:, None].astype("e")), 1)
        for a in a
    ]
).tofile("nnTransferX.35e")
np.stack(a for a in a for a in a[2][1:]).tofile("nnTransferY.30e")

In [ ]:
n = slice(1000)
(
    (
        (
            x[n, None, 30:34] @ x[n, :16].view(-1, 4, 4).transpose(1, 2)
            + x[n, None, 16:20]
        ).tanh()
        @ x[n, 20:28].view(-1, 2, 4).transpose(1, 2)
        + x[n, None, 28:30]
    )[:, 0].argmax(1)
    == x[n, 34]
).sum() / n.stop

In [ ]:
(
    (
        (
            x[n, None, 30:34] @ y[n, :16].view(-1, 4, 4).transpose(1, 2)
            + y[n, None, 16:20]
        ).tanh()
        @ y[n, 20:28].view(-1, 2, 4).transpose(1, 2)
        + y[n, None, 28:30]
    )[:, 0].argmax(1)
    == x[n, 34]
).sum() / n.stop

In [ ]:
r = m(x[n])
(
    (
        (
            x[n, None, 30:34] @ r[n, :16].view(-1, 4, 4).transpose(1, 2)
            + r[n, None, 16:20]
        ).tanh()
        @ r[n, 20:28].view(-1, 2, 4).transpose(1, 2)
        + r[n, None, 28:30]
    )[:, 0].argmax(1)
    == x[n, 34]
).sum() / n.stop